# Portfolio Backtesting

AbaQuant's backtesting layer evaluates periodic simple-return panels,
applies transparent target weights, rebalances on a fixed calendar
schedule, and reports performance, drawdown, turnover, and transaction-cost
diagnostics. It is intentionally **not** an event-driven trading simulator.

**Sections:**
1. Build an allocator and run the object-oriented backtest
2. Run the same backtest through the functional helper
3. Summaries: performance, costs, monthly returns, drawdowns, contributions, trades
4. Visualizations


## Setup


In [ ]:
import os, sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

def _ensure_abaquant_importable():
    try:
        import abaquant  # noqa: F401
        return
    except ModuleNotFoundError:
        pass
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        src = candidate / "src"
        if (src / "abaquant" / "__init__.py").exists():
            sys.path.insert(0, str(src))
            return
    raise ModuleNotFoundError(
        "Could not import abaquant. Run this notebook from within the AbaQuant "
        "repository (or pip install abaquant)."
    )

_ensure_abaquant_importable()
import abaquant
print(f"AbaQuant version: {abaquant.__version__}")

In [ ]:
import numpy as np
import pandas as pd

from abaquant.portfolio import PortfolioAllocator, run_rebalanced_backtest
from abaquant.visualization import VisualizationError

def sample_returns() -> pd.DataFrame:
    dates = pd.date_range("2025-01-02", periods=36, freq="B")
    trend = np.linspace(0.0, 1.0, len(dates))
    seasonal = np.sin(np.linspace(0.0, 4.0 * np.pi, len(dates)))
    prices = pd.DataFrame(
        {
            "ALPHA": 100.0 + 9.0 * trend + 1.8 * seasonal,
            "BETA": 82.0 + 6.0 * trend - 1.2 * seasonal,
            "GAMMA": 54.0 + 3.0 * trend + 0.7 * np.cos(np.linspace(0.0, 3.0 * np.pi, len(dates))),
        },
        index=dates,
    )
    return prices.pct_change().dropna()

## 1. Build an allocator and run the object-oriented backtest

`allocator.backtest(...)` accepts a weight policy (here,
`inverse_volatility`), a rebalance schedule, transaction-cost/slippage
assumptions, and an optional benchmark.


In [ ]:
allocator = PortfolioAllocator(sample_returns(), annual_risk_free_rate=0.02)

allocator_backtest = allocator.backtest(
    weights="inverse_volatility",
    rebalance="monthly",
    transaction_cost_bps=5.0,
    slippage_bps=1.0,
    fixed_transaction_cost=1.0,
    initial_capital=100_000.0,
    benchmark="equal_weight",
    lookback=4,
)
allocator_backtest.summary()

## 2. Run the same backtest through the functional helper

Useful when you want fixed, user-specified weights instead of a policy name.


In [ ]:
functional_backtest = run_rebalanced_backtest(
    sample_returns(),
    weights={"ALPHA": 0.45, "BETA": 0.35, "GAMMA": 0.20},
    rebalance="monthly",
    transaction_cost_bps=5.0,
    slippage_bps=1.0,
    fixed_transaction_cost=1.0,
    initial_capital=100_000.0,
    annual_risk_free_rate=0.02,
    benchmark="equal_weight",
)
functional_backtest.summary()

## 3. Summaries

Cost summary, calendar return table, largest drawdown events, per-asset
contribution summary, per-rebalance trade summary, and rolling metrics.


In [ ]:
allocator_backtest.cost_summary()

In [ ]:
allocator_backtest.return_table()

In [ ]:
allocator_backtest.drawdown_events(top=3)

In [ ]:
allocator_backtest.contribution_summary()

In [ ]:
allocator_backtest.trade_summary()

In [ ]:
allocator_backtest.rolling_metrics(window=4).tail()

## 4. Visualizations

Equity curve, benchmark comparison, drawdown, weights, turnover,
transaction costs, rolling Sharpe/volatility, a calendar return heatmap,
contribution, and trade-weight charts.


In [ ]:
try:
    figures = {
        "equity_curve": allocator_backtest.visualize(chart="equity_curve"),
        "benchmark": allocator_backtest.visualize(chart="benchmark"),
        "drawdown": allocator_backtest.visualize(chart="drawdown"),
        "weights": allocator_backtest.visualize(chart="weights"),
        "turnover": allocator_backtest.visualize(chart="turnover"),
        "transaction_costs": allocator_backtest.visualize(chart="transaction_costs"),
        "rolling_sharpe": allocator_backtest.visualize(chart="rolling_sharpe", rolling_window=4),
        "rolling_volatility": allocator_backtest.visualize(
            chart="rolling_volatility", rolling_window=4
        ),
        "return_heatmap": allocator_backtest.visualize(chart="return_heatmap"),
        "contributions": allocator_backtest.visualize(chart="contributions"),
        "trade_weights": allocator_backtest.visualize(chart="trade_weights"),
    }
    print(f"Created {len(figures)} figures: {list(figures)}")
except VisualizationError as exc:
    print(f"Visualization skipped (optional dependency missing): {exc}")

## Takeaway

Backtests here are deterministic, close-to-close simulations. They do
**not** model intraday execution, taxes, financing, or market impact beyond
the explicit slippage parameter — treat results as directional research,
not a live-trading guarantee. See `docs/domains/assumptions.rst` for the
full list of backtest failure modes.
